# 🎵 Advanced AI Music Studio
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
**Three studios in one:** Generate (Stable Audio) · Remix (MusicGen) · Stem Split (HTDemucs)
Fully optimised with `device_map='auto'` for Colab & Kaggle T4 GPUs.

## 🛠️ Step 1: Environment Setup

In [ ]:
import os
# ── Environment Detection (Kaggle-First) ────────────────────────────────────────
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    ENV = "Kaggle"
else:
    try:
        import google.colab
        ENV = "Colab"
    except ImportError:
        ENV = "Local"

print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Ensure 'Internet' is ON in Kaggle Notebook Settings.")
!pip install -q -U diffusers transformers accelerate bitsandbytes gradio soundfile torchaudio demucs audiocraft
print("✅ Packages ready.")

## 🧠 Step 2: Audio Engines

In [ ]:
import torch, gc, os, numpy as np, soundfile as sf
import gradio as gr, torchaudio, subprocess
from diffusers import StableAudioPipeline
from audiocraft.models import MusicGen

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
SAO_REPO = "stabilityai/stable-audio-open-1.0"
MG_REPO  = "facebook/musicgen-medium"
sao_pipe = None
mg_model = None

def clear_cache():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_sao():
    global sao_pipe
    if sao_pipe is None:
        clear_cache()
        print("⏳ Loading Stable Audio Open...")
        sao_pipe = StableAudioPipeline.from_pretrained(SAO_REPO, torch_dtype=torch.float16, device_map="auto")
        print("✅ Ready.")
    return sao_pipe

def load_mg():
    global mg_model
    if mg_model is None:
        clear_cache()
        print("⏳ Loading MusicGen...")
        mg_model = MusicGen.get_pretrained(MG_REPO)
        print("✅ Ready.")
    return mg_model

def crossfade(a, b, sr, fade=2.0):
    n = int(fade * sr)
    a[-n:] *= np.linspace(1, 0, n)
    b[:n]  *= np.linspace(0, 1, n)
    return np.concatenate([a[:-n], a[-n:]+b[:n], b[n:]])

print("✅ Engine ready.")

## 🎛️ Step 3: Studio Functions

In [ ]:
def studio_generate(prompt, duration, steps, loop, seed):
    if not prompt.strip(): return None, "❌ Prompt required."
    try:
        pipe = load_sao()
        sr   = pipe.vae.sampling_rate
        gen  = torch.Generator(DEVICE)
        if int(seed) != -1: gen.manual_seed(int(seed))
        n_chunks = int(np.ceil(duration / 40))
        audio    = None
        for i in range(n_chunks):
            dur_i = min(47.0, duration - i * 40)
            print(f"  Chunk {i+1}/{n_chunks} ({dur_i}s)...")
            out   = pipe(prompt, num_inference_steps=int(steps), audio_end_in_s=dur_i, generator=gen).audios[0]
            chunk = out.T.float().cpu().numpy().flatten()
            audio = chunk if audio is None else crossfade(audio, chunk, sr)
        if loop: audio = np.concatenate([audio, audio])
        path = "studio_output.wav"
        sf.write(path, audio, int(sr))
        clear_cache()
        return path, f"✅ Generated {duration}s → {path}"
    except Exception as e:
        return None, f"❌ Error: {e}"

def studio_remix(audio_in, prompt, duration):
    if audio_in is None: return None, "❌ Upload a reference track."
    try:
        model = load_mg()
        model.set_generation_params(duration=int(duration))
        melody, sr = torchaudio.load(audio_in)
        out = model.generate_with_chroma([prompt], melody[None].to(DEVICE), sr)
        path = "remix_output.wav"
        torchaudio.save(path, out[0].cpu(), model.sample_rate)
        clear_cache()
        return path, "✅ Remix done!"
    except Exception as e:
        return None, f"❌ Error: {e}"

def studio_split(audio_in):
    if audio_in is None: return None, None, None, None, "❌ Upload a track."
    try:
        subprocess.run(["python3", "-m", "demucs.separate", "-n", "htdemucs", audio_in, "--out", "stems"], check=True)
        base = os.path.splitext(os.path.basename(audio_in))[0]
        d = f"stems/htdemucs/{base}"
        return f"{d}/drums.wav", f"{d}/bass.wav", f"{d}/other.wav", f"{d}/vocals.wav", "✅ Done!"
    except Exception as e:
        return None, None, None, None, f"❌ Error: {e}"

print("✅ Studio functions ready.")

## 🏗️ Step 4: Multi-Tab Studio Interface

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="2M AI Music Studio") as studio:
    gr.Markdown("# 🎹 2M Advanced AI Music Studio\n**Stable Audio Open · MusicGen · HTDemucs · T4 Optimized**")

    with gr.Tab("🎵 Generate"):
        with gr.Row():
            with gr.Column():
                gen_prompt = gr.Textbox(label="Musical Prompt", lines=3, placeholder="Lofi hip hop, chill beats, vinyl crackle, 90 BPM...")
                gen_dur    = gr.Slider(5, 180, value=30, step=5, label="Duration (seconds)")
                gen_steps  = gr.Slider(50, 200, value=100, step=10, label="Diffusion Steps")
                gen_loop   = gr.Checkbox(label="🔁 Seamless Loop")
                gen_seed   = gr.Number(label="Seed (-1 = random)", value=-1)
                gen_btn    = gr.Button("🎵 Generate", variant="primary")
                gen_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column():
                gen_out = gr.Audio(label="Output Track", type="filepath")
        gen_btn.click(studio_generate, [gen_prompt, gen_dur, gen_steps, gen_loop, gen_seed], [gen_out, gen_status])

    with gr.Tab("🌀 Remix"):
        with gr.Row():
            with gr.Column():
                remix_in     = gr.Audio(label="Reference Track", type="filepath")
                remix_prompt = gr.Textbox(label="Style Prompt", placeholder="Transform into orchestral cinematic score...")
                remix_dur    = gr.Slider(5, 60, value=30, step=5, label="Output Duration (s)")
                remix_btn    = gr.Button("🌀 Remix", variant="primary")
                remix_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column():
                remix_out = gr.Audio(label="Remixed Track", type="filepath")
        remix_btn.click(studio_remix, [remix_in, remix_prompt, remix_dur], [remix_out, remix_status])

    with gr.Tab("✂️ Stem Splitter"):
        with gr.Row():
            with gr.Column():
                split_in     = gr.Audio(label="Track to Split", type="filepath")
                split_btn    = gr.Button("✂️ Extract Stems", variant="primary")
                split_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column():
                s_drums  = gr.Audio(label="🥁 Drums",  type="filepath")
                s_bass   = gr.Audio(label="🎸 Bass",   type="filepath")
                s_other  = gr.Audio(label="🎹 Melody", type="filepath")
                s_vocals = gr.Audio(label="🎤 Vocals", type="filepath")
        split_btn.click(studio_split, [split_in], [s_drums, s_bass, s_other, s_vocals, split_status])

studio.launch(share=True, debug=False)

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*